# 03 · Fine-tuned DeBERTa-v3: plain, augmented, and leave-one-family-out

**Needs a GPU** (≈ 40–60 min on an RTX 4070 Super for all runs).

| Run | What it tests |
|---|---|
| `deberta` × 3 seeds | the main sub-word detector, trained on clean data |
| `deberta_aug` × 3 seeds | hardening method 2: random obfuscation of *both* classes during training |
| `deberta_aug_no_<family>` × 6 | trained with all families except one: does augmentation transfer to an obfuscation it never saw? |

Settings (`config.TransformerConfig`): fp32 (bf16 autocast produced NaN losses with DeBERTa-v3),
batch 16, max length 256 for training and 512 for evaluation, AdamW lr 2e-5, 4 epochs, best epoch
chosen on validation F1. Training stops with an error at the first non-finite loss.
Only the seed-42 checkpoints of `deberta` and `deberta_aug` are kept on disk (≈ 0.7 GB each);
all other runs are evaluated straight after training and then discarded.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# os.environ["PI_SMOKE"] = "1"   # uncomment for a 5-minute synthetic test of the whole pipeline

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

from pi_hardening import config
print("smoke mode:", config.SMOKE, "| data:", config.DATA_DIR)

smoke mode: False | data: C:\Users\admin\PycharmProjects\prompt-injection-hardening\data\processed


In [2]:
from dataclasses import asdict, replace
from pi_hardening import data, evaluate
from pi_hardening.detectors.transformer import finetune
from pi_hardening.normalize import Normalizer
from pi_hardening.utils import describe_hardware
print(describe_hardware())
splits = data.load_splits()
bundle = evaluate.load_bundle()
norm = Normalizer()
cfg = config.TransformerConfig()
print(asdict(cfg))
latency_texts = splits["test_in"]["text"].tolist()
latency = []

def run(model_name, seed, augment=None, keep=False):
    """Train, evaluate and save predictions; returns None if this run already has predictions."""
    print(f"== {model_name}  seed {seed}")
    if evaluate.predictions_ok(f"{model_name}__s{seed}"):
        print("   valid predictions already exist - skipped (delete the file to redo)")
        return None
    save_dir = config.MODEL_DIR / f"{model_name}__s{seed}" if keep else None
    det, hist = finetune(splits["train"], splits["val"], cfg, seed=seed, augment=augment,
                         name=model_name, save_dir=save_dir)
    evaluate.save_history(hist, model_name, seed)
    evaluate.save_predictions(evaluate.score_bundle(det, bundle, model=model_name, seed=seed),
                              f"{model_name}__s{seed}")
    return det

CUDA: NVIDIA GeForce RTX 4070 SUPER, 12.0 GB, torch 2.11.0+cu128
{'max_len_train': 256, 'max_len_eval': 512, 'batch_size': 16, 'eval_batch_size': 16, 'lr': 2e-05, 'epochs': 4, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'bf16': False, 'bf16_eval': True, 'seeds': (13, 42, 77)}


## Plain and augmented, three seeds each

In [3]:
aug = config.AugmentConfig()
for model_name, augment in [("deberta", None), ("deberta_aug", aug)]:
    for seed in cfg.seeds:
        det = run(model_name, seed, augment, keep=(seed == 42))
        if det is None:
            continue
        if seed == cfg.seeds[0]:
            evaluate.save_latency([evaluate.measure_latency(det, latency_texts, model_name),
                                   evaluate.measure_latency(det, latency_texts, model_name, normaliser=norm)])
        det.unload()

== deberta  seed 13


W1017 20:10:45.622000 42472 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

  epoch 1: loss 0.3250  val F1 0.9718


...

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

  epoch 1: loss 0.4924  val F1 0.9571


  epoch 2: loss 0.1698  val F1 0.9619


  epoch 3: loss 0.1112  val F1 0.9691


  epoch 4: loss 0.0985  val F1 0.9724


## Leave one family out (seed 42)
The held-out family is never used for augmentation; notebook 05 compares detection on exactly that family.

In [4]:
for held_out in config.FAMILIES:
    fams = [f for f in config.FAMILIES if f != held_out]
    det = run(f"deberta_aug_no_{held_out}", 42, replace(aug, families=fams))
    if det is not None:
        det.unload()

== deberta_aug_no_zero_width  seed 42


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

  epoch 1: loss 0.4377  val F1 0.9580


  epoch 2: loss 0.1860  val F1 0.9724


...

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

  epoch 1: loss 0.4914  val F1 0.9614


  epoch 2: loss 0.1897  val F1 0.9724


  epoch 3: loss 0.1042  val F1 0.9724


  epoch 4: loss 0.0895  val F1 0.9724


In [5]:
m = evaluate.metrics_table(evaluate.load_predictions("deberta*"))
m.query("eval_set == 'test_in' and condition == 'raw'").pivot_table(index="model", columns="family", values="tpr")

family,char_spacing,clean,diacritics,homoglyph,leetspeak,punct_split,zero_width
model,,,,,,,
deberta,0.996,0.959,0.993,0.989,0.984,0.986,0.988
deberta_aug,0.939,0.943,0.934,0.918,0.923,0.936,0.911
deberta_aug_no_char_spacing,0.957,0.947,0.952,0.914,0.909,0.914,0.947
deberta_aug_no_diacritics,0.920,0.941,0.936,0.914,0.936,0.930,0.952
deberta_aug_no_homoglyph,0.963,0.936,0.947,0.930,0.952,0.941,0.936
deberta_aug_no_leetspeak,0.963,0.930,0.973,0.968,0.930,0.968,0.973
deberta_aug_no_punct_split,0.968,0.941,0.952,0.952,0.941,0.925,0.952
deberta_aug_no_zero_width,0.979,0.941,0.909,0.930,0.925,0.936,0.813
